# VIVS: identifying genes that depend on the cellular niche

This tutorial demonstrates [VIVS](https://doi.org/10.1186/s13059-024-03419-z) {cite:p}`Boyeau24`, a conditional randomization test (CRT) that identifies which genes in a cell's expression profile are conditionally dependent on an external response of interest. VIVS learns (or reuses) a deep generative model of gene expression as a calibrated "knockoff" sampler, then tests whether an importance-score network can predict the response `Y` from the *true* expression better than from *knockoff* draws, yielding FDR-controlled p-values per gene (and, via a hierarchical extension, per gene cluster at several resolutions).

Pairing VIVS with scVIVA (see the [scVIVA tutorial](scVIVA_tutorial.ipynb)) is a natural real-data use case: scVIVA already computes, for every cell, a niche-composition vector (the cell-type composition of its spatial neighborhood) together with a generative model of gene expression conditioned on both the cell's own state and its environment. Rather than fitting a fresh VAE from scratch as VIVS's knockoff sampler, we reuse the already-trained scVIVA model directly (via VIVS's `x_model=` argument), and ask: which genes are significantly explained by the surrounding niche composition?

This tutorial assumes you have already gone through the [scVIVA tutorial](scVIVA_tutorial.ipynb) and loads the exact same dataset, so the two can be run back to back.

In [ ]:
import os
import tempfile

import numpy as np
import scanpy as sc
import scviva

scviva.settings.seed = 0
print("Last run with scVIVA-Tools version:", scviva.__version__)

## Data loading

In this tutorial we load a human breast cancer section, generated with [10X Xenium](https://www.nature.com/articles/s41467-023-43458-x).
The cell segmentation originally performed on this data resulted in many erroneously assigned transcripts and therefore re-segmented the cells using the [ProSeg](https://www.biorxiv.org/content/10.1101/2024.04.25.591218v1) algorithm, which is a scalable algorithm for transcriptome-informed segmentation.

In [ ]:
# save_dir / adata loading copied verbatim from scVIVA_tutorial.ipynb so both tutorials
# load the exact same dataset consistently.
save_dir = tempfile.TemporaryDirectory()

adata_path = os.path.join(save_dir.name, "adata_for_tuto_s1.h5ad")
adata = sc.read(
    adata_path,
    backup_url="https://exampledata.scverse.org/scvi-tools/adata_for_tuto_s1.h5ad",
)
adata

## Train a scVIVA model on the niche composition

As in the scVIVA tutorial, we first define the spatial neighborhood of each cell with a k-nearest-neighbor graph ($k=20$). `SCVIVA.preprocessing_anndata` computes, from this graph, the cell-type composition of each cell's neighborhood (`niche_composition`) and the average neighboring gene expression state per cell type (`niche_activation`), and stores both in `adata.obsm`. This assumes a cell-intrinsic expression embedding (e.g. from a previously trained scANVI or resolVI model) is already stored in `adata.obsm["X_scANVI"]`, as in the scVIVA tutorial.

We then register the AnnData and train `scVIVA` to convergence. Its trained module will be reused below as VIVS's knockoff sampler, so VIVS itself will skip fitting its own generative VAE (phase 1) and go directly to testing `niche_composition` dependence (phase 2).

In [ ]:
setup_kwargs = {
    "sample_key": "sample",  # column in adata.obs that contains the individual slide ID
    "labels_key": "cell_type",  # column in adata.obs that contains the cell type labels
    "cell_coordinates_key": "spatial",  # spatial coordinates key in adata.obsm
    "expression_embedding_key": "X_scANVI",  # expression embedding key in adata.obsm
}

scviva.model.SCVIVA.preprocessing_anndata(
    adata,
    k_nn=20,  # number of nearest neighbors for spatial graph construction
    **setup_kwargs,
)

scviva.model.SCVIVA.setup_anndata(
    adata,
    layer="counts",  # adata layer that contains the raw counts
    batch_key="sample",  # column in adata.obs that contains the batch covariate
    **setup_kwargs,
)

scviva_model = scviva.model.SCVIVA(adata)
scviva_model.train(
    max_epochs=600,
    early_stopping=True,
    check_val_every_n_epoch=1,
    batch_size=512,
    plan_kwargs={
        "lr": 5e-4,
    },
)

## Test niche-composition dependence with VIVS

We register the same `adata` for VIVS, with `y_obsm_key="niche_composition"` as the response `Y` whose dependence on gene expression `X` we want to test. Passing `x_model=scviva_model` tells VIVS to reuse the already-trained scVIVA module (frozen) as its knockoff sampler, instead of fitting a new generative VAE — VIVS's `.train()` call then only fits the importance-score network for `Y | X`.

In [ ]:
scviva.model.VIVS.setup_anndata(
    adata,
    y_obsm_key="niche_composition",  # scVIVA's niche cell-type-composition vector, in adata.obsm
    layer="counts",  # same raw-count layer used to train scVIVA
    batch_key="sample",  # column in adata.obs that contains the batch covariate
)

vivs_model = scviva.model.VIVS(adata, x_model=scviva_model)
vivs_model.train(max_epochs=200)

## Hierarchical gene importance

`get_hier_importance` clusters genes by decoder-scale correlation at several resolutions, then re-runs the conditional randomization test with group-level knockoff substitution at each resolution, giving FDR-controlled p-values both for individual genes and for coarser gene clusters. Passing `n_clusters_list=[50, 100, 200]` tests three resolutions in addition to the finest (per-gene) one.

In [ ]:
res = vivs_model.get_hier_importance(n_clusters_list=[50, 100, 200])
res

## Visualizing results

`plot_hier_importance` renders the multi-resolution results as a significance dendrogram: genes/clusters found significant (BH-adjusted p-value below `significance_threshold`) at the coarsest resolution are shown, colored by significance, across all tested resolutions.

In [ ]:
scviva.pl.plot_hier_importance(res, theme_kwargs=dict(figure_size=(15, 3)))

## Interpretation and next steps

A gene (or gene cluster) called significant at a given resolution means the CRT rejected the null hypothesis that gene expression is conditionally independent of the niche composition `Y`, at the chosen FDR level, when knockoffs are drawn from scVIVA's generative model — i.e. that gene's expression carries information about the surrounding niche beyond what is already explained by every other gene. Coarser resolutions (larger gene clusters) aggregate evidence across correlated genes and tend to have more power to detect weaker, distributed niche effects, while the finest (per-gene) resolution pinpoints individual genes.

For a per-cell rather than per-dataset view of which cells drive a given gene's (or gene cluster's) importance score, see {meth}`~scviva.model.VIVS.get_cell_scores`, which returns unsummed, per-cell importance scores for a chosen set of genes and responses.